# Driver and Safety Analysis

## Goal

1. Who are the drivers, and how long have they been with the company?
2. Does on-time rate or fuel efficiency differ by driver tenure/experience?
3. Do individual drivers stand out, or is the spread consistent with chance?
4. What do safety incident rates look like, and do they relate to driver experience?

## Data Basis

`drivers`, `driver_monthly_metrics`, `trips`, `fuel_purchases` (only for the `driver_id` NULL check),
`safety_incidents`, plus delivery metrics
from the pipeline. Note: `safety_incidents` has only 170 records - treat rates and
comparisons with appropriate caution given the small sample.

## 1. Load Data (via Pipeline)

In [ ]:
import sys

sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

from delivery_pipeline import PROCESSED_DATA_PATH_DRIVER, RAW_DATA_DIR, run_pipeline

clean_df = run_pipeline(output_path=PROCESSED_DATA_PATH_DRIVER, include_drivers=True)
drivers_df = pd.read_csv(RAW_DATA_DIR / "drivers.csv")
driver_monthly_metrics_df = pd.read_csv(RAW_DATA_DIR / "driver_monthly_metrics.csv")
safety_incidents_df = pd.read_csv(RAW_DATA_DIR / "safety_incidents.csv")

display(clean_df.shape)
display(drivers_df.shape)
display(driver_monthly_metrics_df.shape)
display(safety_incidents_df.shape)

## 2. Data Quality Checks

In [ ]:
# TODO: NULL share of driver_id in trips/fuel_purchases (~2% expected, per dataset description); orphan checks
# TODO: safety_incidents: NULL driver_id/truck_id (1 each); compare driver_id/truck_id with the linked trip (via trip_id);
#       how many incidents link to trips that are still in the cleaned data?

# NULL share of driver_id in the load table (expected ~2%, intentional per dataset description)
print(f"Loads without driver_id: {clean_df['driver_id'].isna().mean():.4%}")

# fuel_purchases is in the Data Basis only for this check
fuel = pd.read_csv(RAW_DATA_DIR / "fuel_purchases.csv")
print(f"Fuel purchases without driver_id: {fuel['driver_id'].isna().mean():.4%}")

# Orphans and duplicates
known = drivers_df["driver_id"]
print(
    "Loads with unknown driver_id:",
    (clean_df["driver_id"].notna() & ~clean_df["driver_id"].isin(known)).sum(),
)
print("Duplicate driver_ids in drivers:", known.duplicated().sum())
print(
    "Incidents with unknown driver_id:",
    (
        safety_incidents_df["driver_id"].notna()
        & ~safety_incidents_df["driver_id"].isin(known)
    ).sum(),
)
print(
    "Incidents without driver_id / truck_id:",
    safety_incidents_df["driver_id"].isna().sum(),
    safety_incidents_df["truck_id"].isna().sum(),
)

In [ ]:
# trip_id can be NaN for loads without a trip; drop those so that the right side is unique
trip_info = clean_df.dropna(subset=["trip_id"])[["trip_id", "driver_id", "truck_id"]]
linked = safety_incidents_df.merge(
    trip_info,
    on="trip_id",
    how="left",
    suffixes=("", "_trip"),
    validate="many_to_one",
    indicator=True,
)
print(
    linked["_merge"].value_counts()
)  # how many incidents link to a trip in the cleaned data?

both = (
    linked["_merge"].eq("both")
    & linked["driver_id"].notna()
    & linked["driver_id_trip"].notna()
)
print(
    "driver_id matches the trip:",
    (linked.loc[both, "driver_id"] == linked.loc[both, "driver_id_trip"]).mean(),
)
both_t = (
    linked["_merge"].eq("both")
    & linked["truck_id"].notna()
    & linked["truck_id_trip"].notna()
)
print(
    "truck_id matches the trip:",
    (linked.loc[both_t, "truck_id"] == linked.loc[both_t, "truck_id_trip"]).mean(),
)

### Data Quality Note

Checked: share of loads without `driver_id`, orphan keys, and whether incidents link to trips of the cleaned data. Result: 2 % of loads have no `driver_id` (the dataset description states about 2 % as intentional); 170 incidents of 170 link to a trip in the cleaned data; `driver_id` of the incident matches the trip in 100 % of cases.

## 3. Driver Overview

### 3.1 Tenure at trip date

*TODO: tenure is measured at the time of each trip (trip date minus hire date), not as of today, because the data covers 2022-2024.*

In [ ]:
# TODO: derive tenure in years at trip date (do not round); check for negative values (trip before hire date)
# and for trips after the termination date

### 3.2 Driver distribution

*TODO: 150 drivers in the table, but only the 124 still employed appear in the trips - mention this.*

In [ ]:
# TODO: driver tenure/employment history distribution, CDL info if present

In [ ]:
# TODO: chart(s)

*TODO: interpretation.*

## 4. Performance by Driver Tenure

### 4.0 Tenure buckets

In [ ]:
# TODO: define buckets with pd.cut (boundaries after looking at the distribution from section 3),
# show the number of trips per bucket (and how many trips have no bucket)

### 4.1 Does on-time rate differ by driver tenure?

In [ ]:
# TODO: groupby tenure bucket -> on_time_flag mean / delay rate

### 4.2 Does fuel efficiency differ by driver tenure?

In [ ]:
# TODO: groupby tenure bucket -> mean average_mpg (via trips -> driver_id)

In [ ]:
# TODO: chart(s)

### 4.3 Is the difference more than chance?

In [ ]:
# TODO: permutation test on the bucket labels (permutation_spread), for the delay rate and average_mpg

*TODO: interpretation.*

## 5. Do Individual Drivers Stand Out?

In [ ]:
# TODO: per-driver delay rate (minimum trip count, justify the threshold) + permutation test,
# same approach as notebooks 3/4/5 (permutation_spread, seed 42, 5,000 iterations)

In [ ]:
# TODO: plot

*TODO: interpretation.*

## 6. Driver Monthly Metrics

### 6.1 How does driver turnover develop over time?

In [ ]:
# TODO: active drivers per month, hires and terminations 2022-2024 from the drivers table (hire_date / termination_date),
# turnover per year (define it in the text); driver_monthly_metrics only holds the drivers still employed, so it carries no turnover information

*TODO: interpretation - dataset description states ~15% annual turnover; compare.*

### 6.2 What does driver_monthly_metrics contain?

In [ ]:
# TODO: only still-employed drivers (124 x 36 months); compare trips_completed / total_miles with the trips table
# (sums, never average the rates); survivorship caveat for any analysis on this table

*TODO: interpretation.*

## 7. Safety Incidents

### 7.1 What do incident rates look like?

In [ ]:
# TODO: incidents per million miles (overall): count only incidents linked to trips in the cleaned data,
# miles from the same trips; shares of at_fault_flag / preventable_flag / injury_flag; breakdown by incident_type

### 7.2 Do incident rates relate to driver experience?

In [ ]:
# TODO: incidents AND miles driven per tenure bucket -> incident rate per million miles
# (counts alone mislead because buckets differ in exposure); caution: only 170 incidents total (fewer after linking to cleaned trips)

In [ ]:
# TODO: chart(s)

*TODO: interpretation - be explicit about sample size limitations here.*

## 8. Conclusion

*TODO: summarize (findings and ideas for later kept apart); mention notebook 7 as an outlook, no link until it exists.*